# Phase 1: ingestion stats
Bronze (raw, HDFS JSON) vs silver (Delta: clean text + chunks). Documents per source, average length, languages, chunk sizes.

Run headless with `make test-phase1`, or open in Jupyter after `make crawl && make silver`.

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("phase1-stats").getOrCreate()
HDFS = os.environ.get("HDFS_URL", "hdfs://namenode:9000")
bronze = spark.read.json(f"{HDFS}/datalake/bronze/*/*/*.json").select("id", "source", "content_type")
docs = spark.read.format("delta").load(f"{HDFS}/datalake/silver/documents")
chunks = spark.read.format("delta").load(f"{HDFS}/datalake/silver/chunks")
pd.set_option("display.width", 160)

In [ ]:
per_source = (docs.groupBy("source").agg(
        F.count("*").alias("silver_docs"),
        F.sum(F.col("is_english").cast("int")).alias("english"),
        F.round(F.avg("n_chars")).cast("int").alias("avg_chars"),
        F.round(F.avg("n_words")).cast("int").alias("avg_words"),
        F.round(F.avg("n_tokens_est")).cast("int").alias("avg_tokens_est"),
        F.expr("percentile_approx(n_words, 0.5)").alias("median_words"))
    .join(bronze.groupBy("source").count().withColumnRenamed("count", "bronze_docs"), "source", "outer")
    .join(chunks.groupBy("source").agg(F.count("*").alias("chunks"),
                                       F.round(F.avg("n_tokens_est")).cast("int").alias("avg_chunk_tokens")),
          "source", "left")
    .orderBy("source").toPandas())
cols = ["source", "bronze_docs", "silver_docs", "english", "avg_chars", "avg_words", "median_words",
        "avg_tokens_est", "chunks", "avg_chunk_tokens"]
per_source = per_source[cols].fillna(0)
total = per_source[["bronze_docs", "silver_docs", "english", "chunks"]].sum()
print(per_source.to_string(index=False))
print(f"\nTOTAL bronze={int(total.bronze_docs)} silver={int(total.silver_docs)} "
      f"english={int(total.english)} chunks={int(total.chunks)} sources={len(per_source)}")

In [ ]:
langs = docs.groupBy("language").count().orderBy(F.desc("count")).toPandas()
print("Languages:", ", ".join(f"{r.language}={r['count']}" for _, r in langs.iterrows()))
errors = docs.filter(F.col("extract_error").isNotNull()).select("source", "title", "extract_error").toPandas()
print(f"Extraction errors: {len(errors)}")
if len(errors):
    print(errors.to_string(index=False))
empty = docs.filter(F.col("n_words") < 20).select("source", "title", "n_words").toPandas()
print(f"Docs with < 20 words (not chunked): {len(empty)}")

In [ ]:
lengths = docs.select("source", "n_words").toPandas()
chunk_tokens = chunks.select("n_tokens_est").toPandas()
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
per_source.plot.bar(x="source", y=["bronze_docs", "chunks"], ax=ax[0], rot=0, title="Documents and chunks per source")
for src, g in lengths.groupby("source"):
    ax[1].hist(g.n_words.clip(lower=1), bins=30, alpha=0.6, label=src, log=True)
ax[1].set(title="Document length (words)", xlabel="words", ylabel="docs (log)"); ax[1].legend()
ax[2].hist(chunk_tokens.n_tokens_est, bins=30)
ax[2].set(title="Chunk size (estimated tokens)", xlabel="tokens (≈ words / 0.75)", ylabel="chunks")
fig.tight_layout()
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/phase1_ingestion_stats.png", dpi=110)
print("saved figures/phase1_ingestion_stats.png")
print(f"STATS_OK bronze={int(total.bronze_docs)} chunks={int(total.chunks)} sources={len(per_source)}")
spark.stop()